# Item pipelines

> Running all pipelines for one item and inspect the changes.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections.abc import Generator
from pathlib import Path

import polars as pl
import scrapy
from IPython.display import display as ipython_display
from pydantic import ValidationError
from scrapy.crawler import AsyncCrawlerRunner
from scrapy.http import XmlResponse
from scrapy.settings import Settings
from scrapy.utils.project import get_project_settings

from notebooks.render_pipeline import pipeline_trace_viewer
from notebooks.trace_pipeline import trace, trace_pipeline
from pazufa_scraper_be.pardok.dokument import DrsDokument
from pazufa_scraper_be.pardok.vorgang import GesetzVorgang
from pazufa_scraper_be.pipelines import (
    BuildPaZuFaVorgang,
    DownloadDocuments,
    ExtractText,
    HandleErrors,
    PrepareDocumentUrls,
    SummarizeExtractedPDFText,
)
from pazufa_scraper_be.spiders.utils import convert_element_to_dict


class ManualInputSpider(scrapy.Spider):
    """Manual spider using a static item to pass to the pipelines."""

    name = "manual-input"

    def __init__(self, item: GesetzVorgang | dict, name: str | None = None, **kwargs: object) -> None:
        """Spider for `item`, a manually constructed GesetzVorgang."""
        super().__init__(name=name, **kwargs)
        self.item = item

    async def start(self) -> Generator[GesetzVorgang | dict]:  # ty: ignore[invalid-method-override, invalid-return-type]
        """Start with fix item to pass to the pipelines."""
        yield self.item


async def run_test(settings: Settings, item: GesetzVorgang | dict) -> None:
    """Run pipelines with dummy spider."""
    runner = AsyncCrawlerRunner(settings=settings)
    await runner.crawl(ManualInputSpider, item=item)

In [ ]:
item_pipelines = {
    HandleErrors: 99,
    PrepareDocumentUrls: 100,
    DownloadDocuments: 110,
    ExtractText: 120,
    SummarizeExtractedPDFText: 130,
    BuildPaZuFaVorgang: 999,
}

In [ ]:
for pipeline_cls in item_pipelines:
    trace_pipeline(pipeline_cls)

In [ ]:
settings = get_project_settings()
settings["ITEM_PIPELINES"] = item_pipelines
# twisted reactor not needed in notebooks, skipping the reactor though so some behavior may be different
settings["TWISTED_REACTOR_ENABLED"] = False
settings

In [ ]:
def make_test_vorgang() -> GesetzVorgang:
    """Create a dummy Vorgang."""
    dokument = DrsDokument(
        art="Drs",
        art_l="Drucksache",
        herk="Pardok",
        herk_l="Pardok-System",
        typ="Antr",
        typ_l="Antrag",
        id="doc123",
        wp=19,
        reih_nr=1,
        nr="123/2024",
        dat="10.05.2024",
        lok_url="https://example.invalid/doc123.pdf",
        titel="Entwurf eines Testgesetzes",
        urheber=["Testfraktion"],
    )

    return GesetzVorgang(
        typ="Gesetz",
        typ_l="Gesetz",
        id="vorg123",
        reih_nr=0,
        nr="V123/2024",
        ir="IR123",
        sys=[1000],
        sys_l=["Test"],
        dokumente=[dokument],
    )


item = make_test_vorgang()
item

In [ ]:
cache_dir: Path = settings.get("CACHE_DIR")
feeds = list(cache_dir.glob("**/*.xml"))
feeds

In [ ]:
def get_actual_vorgang(feed: Path, vorgang_idx: int) -> GesetzVorgang | dict:
    """Produce object for an actual Vorgang entry."""
    response = XmlResponse(url=feed.resolve().as_uri(), body=feed.read_bytes(), encoding="utf-8")
    gesetz_vorgang_dicts = [convert_element_to_dict(x) for x in response.selector.xpath("/Export/Vorgang[./VTyp[text() = 'Gesetz']]")]

    try:
        g = GesetzVorgang.model_validate(gesetz_vorgang_dicts[vorgang_idx])
    except ValidationError as error:
        g = {"dict": gesetz_vorgang_dicts[vorgang_idx], "error": error}

    return g


feed_to_use = feeds[0]
vorgang_idx = 0

item = get_actual_vorgang(feed_to_use, vorgang_idx)
item

In [ ]:
await run_test(settings, item=item)

In [ ]:
trace_df = pl.from_dicts(trace)
trace_df.head(2)

In [ ]:
ipython_display(pipeline_trace_viewer(trace))  # ty: ignore[invalid-argument-type]